# 01 — Environnement DGX Spark et données
**Projet :** assistant de service client e-commerce (LLM Engineer) · **Objectif :** prouver l'environnement (GPU, LLM servi) et contrôler la qualité de la base SQLite.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
pd.set_option("display.max_colwidth", 200); pd.set_option("display.width", 220)

## 1. Environnement matériel (DGX Spark)

In [2]:
import subprocess, platform, socket
print("Hôte :", socket.gethostname(), "|", platform.platform(), "| Python", platform.python_version())
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

Hôte : spark-4001 | Linux-7.0.0-1019-nvidia-aarch64-with-glibc2.39 | Python 3.12.3
Tue Oct 13 14:38:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GB10                    On  |   0000000F:01:00.0 Off |                  N/A |
| N/A   55C    P0             13W /  N/A  | Not Supported          |      0%      Default |
|                                         |              

## 2. LLM servi localement (vLLM, API compatible OpenAI)

In [3]:
import json, urllib.request
from assistant.config import LLM_BASE_URL
models = json.load(urllib.request.urlopen(LLM_BASE_URL + "/models"))["data"]
print("Endpoint :", LLM_BASE_URL); print("Modèle servi :", models[0]["id"], "| contexte max :", models[0]["max_model_len"])
from assistant.llm import get_llm
print("Test LLM :", get_llm().invoke("Réponds uniquement par : OK").content)

Endpoint : http://localhost:8000/v1
Modèle servi : unsloth/Qwen3.8-27B-NVFP4 | contexte max : 131072


Test LLM : OK


In [4]:
import langchain, langchain_openai, sentence_transformers
print("langchain", langchain.__version__, "| langchain-openai", langchain_openai.__version__, "| sentence-transformers", sentence_transformers.__version__)

.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


langchain 1.4.3 | langchain-openai 1.6.7 | sentence-transformers 6.1.0


## 3. Chargement de la base SQLite

In [5]:
import sqlite3
from assistant.config import DB_PATH
con = sqlite3.connect(DB_PATH)
users = pd.read_sql("SELECT * FROM users", con); orders = pd.read_sql("SELECT * FROM orders", con)
print("Tables :", [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")])
print(f"users : {len(users)} lignes | orders : {len(orders)} lignes")
users.head(3)

Tables : ['orders', 'users']
users : 40 lignes | orders : 100 lignes


,index,user_id,first_name,last_name,joining_date,phone,email,address,city,zip_code
0,0,1,Ramona,Howell,2024-05-27 14:15:52,686855635,at@google.com,6096 Inceptos Ave,Le Puy-en-Velay,78971
1,1,2,Destiny,Hendrix,2024-01-19 11:07:01,678832573,mollis.non@aol.com,Ap #783-2387 Sodales Rd.,Niort,25744
2,2,3,Hadassah,May,2024-05-31 08:34:10,688426254,massa@protonmail.com,"P.O. Box 530, 2177 Aliquam Rd.",La Roche-sur-Yon,16789


In [6]:
orders.head(5)

,index,order_id,user_id,status,date_purchase,date_shipped,date_delivered
0,0,1,32,shipped,2024-05-25 22:27:26,2024-05-26 22:27:26,NaN
1,1,2,24,delivered,2024-05-06 12:57:32,2024-05-08 12:57:32,2024-05-22 12:57:32
2,2,3,17,delivered,2024-05-08 05:05:25,2024-05-09 05:05:25,2024-05-20 05:05:25
3,3,4,16,delivered,2024-05-18 20:38:31,2024-05-19 20:38:31,2024-05-29 20:38:31
4,4,5,6,delivered,2024-05-17 11:01:51,2024-05-18 11:01:51,2024-05-28 11:01:51


## 4. Contrôles qualité (dossier §5.4)

In [7]:
checks = {"40 utilisateurs": len(users)==40, "100 commandes": len(orders)==100,
          "user_id uniques": users.user_id.is_unique, "order_id uniques": orders.order_id.is_unique, "emails uniques": users.email.is_unique}
for k, v in checks.items(): print("OK " if v else "KO ", k)
print("\nRépartition des statuts :"); print(orders.status.value_counts())
print("\nDates manquantes par statut :")
print(orders.groupby("status")[["date_shipped","date_delivered"]].agg(lambda s: s.isna().sum()))

OK  40 utilisateurs
OK  100 commandes
OK  user_id uniques
OK  order_id uniques
OK  emails uniques

Répartition des statuts :
status
delivered    55
shipped      37
invoiced      8
Name: count, dtype: int64

Dates manquantes par statut :
           date_shipped  date_delivered
status                                 
delivered             0               0
invoiced              8               8
shipped               0              37


In [8]:
anom = orders[~orders.user_id.isin(users.user_id)]
print("Commandes dont le user_id n'existe pas dans users (anomalies) :"); display(anom)
print("=> Ces commandes ne sont rattachées à aucun client : elles ne seront jamais renvoyées (SQL filtré par user_id authentifié).")

Commandes dont le user_id n'existe pas dans users (anomalies) :


,index,order_id,user_id,status,date_purchase,date_shipped,date_delivered
53,53,54,0,shipped,2024-05-25 21:54:59,2024-05-27 21:54:59,NaN
70,70,71,0,shipped,2024-05-28 17:15:40,2024-05-30 17:15:40,NaN
77,77,78,0,delivered,2024-05-14 04:54:18,2024-05-16 04:54:18,2024-05-23 04:54:18


=> Ces commandes ne sont rattachées à aucun client : elles ne seront jamais renvoyées (SQL filtré par user_id authentifié).


In [9]:
print("Commandes par client (top 5) :"); print(orders[orders.user_id>0].groupby("user_id").size().sort_values(ascending=False).head())
print("Clients sans commande :", (~users.user_id.isin(orders.user_id)).sum())

Commandes par client (top 5) :
user_id
23    5
4     5
25    5
9     5
10    5
dtype: int64
Clients sans commande : 1


**Conclusion :** environnement GPU et LLM local opérationnels ; base conforme (40/100), 3 anomalies `user_id=0` (commandes 54, 71, 78) identifiées et neutralisées par conception ; dates absentes cohérentes avec les statuts (invoiced → ni expédition ni livraison ; shipped → pas de livraison).